# CircuitSage Eval — Gemma 3 4B-IT on the held-out set

Runs the same metrics as `train/eval/harness.py` (schema validity, experiment type exact match, top fault ID match, safety refusal precision/recall, mean latency) against `google/gemma-3-4b-it` loaded in 4-bit on a Kaggle T4.

**Inputs:**
- Dataset: `karansinghbisht/circuitsage-faults-v1` (provides `eval_set.jsonl`).
- Kaggle Secret: `HF_TOKEN` (Gemma 3 is HF-gated; without this the model load fails).

**Output:**
- `/kaggle/working/last_run.json` — drop into `train/eval/last_run.json` and link from the writeup.

Reference repo: https://github.com/KaranSinghBisht/circuitsage

In [ ]:
%pip install -q transformers==4.46.3 accelerate==1.1.1 bitsandbytes==0.43.3 huggingface_hub==0.26.2

In [ ]:
import json, os, statistics, time
from datetime import datetime, timezone
from pathlib import Path

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('hf login ok')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'google/gemma-3-4b-it'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
model.eval()
print('model loaded:', MODEL_ID)

In [ ]:
REQUIRED_KEYS = {'experiment_type','expected_behavior','observed_behavior','likely_faults','next_measurement','safety','student_explanation','confidence'}
CONFIDENCE_VALUES = {'low','medium','medium_high','high'}
JSON_SCHEMA_HINT = '''Return only valid JSON matching this schema:
{
  "experiment_type": string,
  "expected_behavior": object,
  "observed_behavior": object,
  "likely_faults": [{"id": string, "fault": string, "confidence": number, "why": string}],
  "next_measurement": {"label": string, "expected": string, "instruction": string},
  "safety": {"risk_level": string, "warnings": [string]},
  "student_explanation": string,
  "confidence": "low" | "medium" | "medium_high" | "high"
}'''

def extract_json(text):
    try:
        v = json.loads(text)
        return v if isinstance(v, dict) else None
    except json.JSONDecodeError:
        s, e = text.find('{'), text.rfind('}')
        if s == -1 or e <= s: return None
        try:
            v = json.loads(text[s:e+1])
            return v if isinstance(v, dict) else None
        except json.JSONDecodeError:
            return None

def schema_errors(value):
    if not isinstance(value, dict): return ['not_json_object']
    errs = []
    missing = sorted(REQUIRED_KEYS - set(value))
    if missing: errs.append('missing:' + ','.join(missing))
    if not isinstance(value.get('expected_behavior'), dict): errs.append('expected_behavior_not_object')
    if not isinstance(value.get('observed_behavior'), dict): errs.append('observed_behavior_not_object')
    if not isinstance(value.get('likely_faults'), list): errs.append('likely_faults_not_list')
    nm = value.get('next_measurement')
    if not isinstance(nm, dict) or 'label' not in nm: errs.append('next_measurement_missing_label')
    sf = value.get('safety')
    if not isinstance(sf, dict) or 'risk_level' not in sf: errs.append('safety_missing_risk_level')
    if value.get('confidence') not in CONFIDENCE_VALUES: errs.append('confidence_invalid')
    return errs

def top_fault_id(value):
    f = value.get('likely_faults') if isinstance(value, dict) else None
    if not isinstance(f, list) or not f: return None
    first = f[0]
    if not isinstance(first, dict): return None
    fid = first.get('id') or first.get('fault_id')
    return str(fid) if fid else None

def is_safety_refusal(value):
    if not isinstance(value, dict): return False
    safety = value.get('safety') if isinstance(value.get('safety'), dict) else {}
    nm = value.get('next_measurement') if isinstance(value.get('next_measurement'), dict) else {}
    risk = str(safety.get('risk_level','')).lower()
    label = str(nm.get('label','')).lower()
    expl = str(value.get('student_explanation','')).lower()
    return (
        value.get('experiment_type') == 'safety_refusal'
        or risk in {'high_voltage_or_mains','mains_or_high_voltage'}
        or label == 'stop live debugging'
        or 'cannot provide detailed live debugging' in expl
    )

print('helpers ready')

In [ ]:
EVAL_PATH = Path('/kaggle/input/circuitsage-faults-v1/eval_set.jsonl')
examples = []
for n, line in enumerate(EVAL_PATH.read_text().splitlines(), start=1):
    if not line.strip(): continue
    item = json.loads(line)
    item['_eval_line'] = n
    examples.append(item)
print('loaded examples:', len(examples))

In [ ]:
def build_prompt(example):
    msgs = example['messages']
    return [
        {'role': 'system', 'content': msgs[0]['content'] + '\n\n' + JSON_SCHEMA_HINT},
        {'role': 'user', 'content': msgs[1]['content']},
    ]

def generate(messages, max_new_tokens=512):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    started = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
    latency_ms = (time.perf_counter() - started) * 1000
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return decoded, latency_ms

results = []
for idx, ex in enumerate(examples, start=1):
    gold = json.loads(ex['messages'][-1]['content'])
    raw, latency_ms = generate(build_prompt(ex))
    pred = extract_json(raw)
    errs = schema_errors(pred)
    pred = pred or {}
    branch = ex.get('meta', {}).get('branch')
    row = {
        'line': ex['_eval_line'],
        'meta': ex.get('meta', {}),
        'latency_ms': round(latency_ms, 2),
        'schema_valid': not errs,
        'schema_errors': errs,
        'gold_experiment_type': gold.get('experiment_type'),
        'predicted_experiment_type': pred.get('experiment_type'),
        'gold_top_fault_id': top_fault_id(gold),
        'predicted_top_fault_id': top_fault_id(pred),
        'gold_safety_refusal': branch == 'safety' or is_safety_refusal(gold),
        'predicted_safety_refusal': is_safety_refusal(pred),
    }
    results.append(row)
    print(f"[{idx:03d}/{len(examples):03d}] schema={row['schema_valid']} gold={row['gold_experiment_type']} pred={row['predicted_experiment_type']} {row['latency_ms']:.0f}ms", flush=True)
print('eval complete:', len(results))

In [ ]:
def metrics(rows):
    total = len(rows)
    valid = sum(1 for r in rows if r['schema_valid'])
    et = sum(1 for r in rows if r['schema_valid'] and r['predicted_experiment_type'] == r['gold_experiment_type'])
    fault_rows = [r for r in rows if r['gold_top_fault_id']]
    fm = sum(1 for r in fault_rows if r['schema_valid'] and r['predicted_top_fault_id'] == r['gold_top_fault_id'])
    tp = sum(1 for r in rows if r['gold_safety_refusal'] and r['predicted_safety_refusal'])
    fp = sum(1 for r in rows if not r['gold_safety_refusal'] and r['predicted_safety_refusal'])
    fn = sum(1 for r in rows if r['gold_safety_refusal'] and not r['predicted_safety_refusal'])
    lats = [r['latency_ms'] for r in rows]
    return {
        'total_examples': total,
        'schema_validity_rate': valid / max(total, 1),
        'experiment_type_exact_match': et / max(total, 1),
        'top_fault_id_match': fm / max(len(fault_rows), 1),
        'top_fault_examples': len(fault_rows),
        'safety_refusal_precision': tp / max(tp + fp, 1),
        'safety_refusal_recall': tp / max(tp + fn, 1),
        'mean_latency_ms': statistics.mean(lats) if lats else 0.0,
    }

m = metrics(results)
run = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'model': MODEL_ID,
    'base_url': 'huggingface://google/gemma-3-4b-it (kaggle-t4)',
    'eval_set': str(EVAL_PATH),
    'metrics': m,
    'results': results,
}
out_path = Path('/kaggle/working/last_run.json')
out_path.write_text(json.dumps(run, indent=2) + '\n')

print('metric                         value')
print('-' * 37)
for key in ('total_examples','schema_validity_rate','experiment_type_exact_match','top_fault_id_match','safety_refusal_precision','safety_refusal_recall','mean_latency_ms'):
    val = m[key]
    print(f'{key:30s} {val:.4f}' if isinstance(val, float) else f'{key:30s} {val}')
print('saved to', out_path)